In [62]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import torch.optim as optimizer

import seaborn as sns

In [63]:
train_val = pd.read_csv('data/train_values.csv') #The data values of each building
train_labels = pd.read_csv('data/train_labels.csv') #labels of data as damage severity ('damage_grade')

test_val = pd.read_csv('data/test_values.csv') #data values of contest submition (our AI would have to guess this, 
#but we won't know their 'damage_grade' for sure until DrivenData releases the answers.

#We combine the features and target label for data sampling and other transformations
df = pd.merge(train_val, train_labels, on='building_id')

#with neural networks, we need damage grade to be 0, 1, 2, instead of 1, 2, 3
#I need to remember to correct for this when submitting
df['damage_grade'] = df['damage_grade'] - 1


In [64]:
#This will state whether neural network is set up to use GPU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [81]:
#Sampling the data so test runs don't take so long
dataSample = df.sample(frac=1)
#define X (data features) and y (target)
#X is all useful features
#y is target (final column 'default payment next month')

#Default Features to drop:
#building_id: for recordkeeping rather than data
#The target variable: damage_grade
features = dataSample.drop(['building_id', 'damage_grade'], axis = 1)

#Other features to drop:
# dropColumns = ['geo_level_1_id', 'geo_level_2_id', 'geo_level_3_id'] #Too specific categorical, would produce thousands of columns
#I decided we keep geo_level_1_id as there are only 30 and it is overal region, may have more value
#but we need to make it categorical to generate one-hot encodings
# features['geo_level_1_id'] = features['geo_level_1_id'].astype('category')

# features = features.drop(dropColumns, axis = 1)
#Data adjustment
# encoder = OneHotEncoder(sparse_output=False, drop='first')

#These tables are in string type and must be converted
# categoryTables = ['geo_level_1_id','land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'legal_ownership_status']
categoryTables = ['land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'legal_ownership_status']
one_hot_tables = pd.get_dummies(features[categoryTables], dtype=int, drop_first=True)




#Now we replace our string tables with the encoded tables
X = features.drop(categoryTables, axis = 1)
X = X.join(one_hot_tables)
print(X.info())

# scaler = MinMaxScaler()
scaler = StandardScaler()
X = scaler.fit_transform(X)


y = dataSample['damage_grade']

<class 'pandas.core.frame.DataFrame'>
Index: 260601 entries, 74415 to 220312
Data columns (total 60 columns):
 #   Column                                  Non-Null Count   Dtype
---  ------                                  --------------   -----
 0   geo_level_1_id                          260601 non-null  int64
 1   geo_level_2_id                          260601 non-null  int64
 2   geo_level_3_id                          260601 non-null  int64
 3   count_floors_pre_eq                     260601 non-null  int64
 4   age                                     260601 non-null  int64
 5   area_percentage                         260601 non-null  int64
 6   height_percentage                       260601 non-null  int64
 7   has_superstructure_adobe_mud            260601 non-null  int64
 8   has_superstructure_mud_mortar_stone     260601 non-null  int64
 9   has_superstructure_stone_flag           260601 non-null  int64
 10  has_superstructure_cement_mortar_stone  260601 non-null  int64
 11  h

In [82]:
y.head(40)

74415     1
168108    2
133976    1
226731    1
152490    2
53962     2
165599    1
256267    1
66556     1
135817    1
192327    2
220762    1
245937    1
2357      1
68753     2
26205     1
226637    1
170194    1
254513    1
19693     1
41762     1
1823      1
163655    0
226638    1
132488    1
153955    1
208628    1
241597    1
160830    0
47113     1
199492    1
248968    1
42491     1
8901      2
129442    1
79294     2
26886     1
119241    1
64765     1
111032    1
Name: damage_grade, dtype: int64

In [83]:
#splitting data and 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [84]:
y_train

151639    2
98762     2
254724    1
204175    1
28611     1
         ..
62900     1
243782    1
170305    2
134012    2
235750    1
Name: damage_grade, Length: 208480, dtype: int64

In [85]:
# X_train.size(0)

In [86]:


X_train_tensor = torch.tensor(X_train)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)

X_test_tensor = torch.tensor(X_test)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

In [87]:
print(X_train_tensor)
print(X_train_tensor.size())

tensor([[-0.7345, -1.4685, -0.4042,  ..., -0.0754,  0.1962, -0.1019],
        [ 0.8837, -0.9403,  1.4947,  ..., -0.0754,  0.1962, -0.1019],
        [-0.4855, -0.7174, -0.3088,  ..., -0.0754,  0.1962, -0.1019],
        ...,
        [-0.9834,  0.3487,  0.2381,  ..., -0.0754,  0.1962, -0.1019],
        [-0.7345, -0.5769, -1.0618,  ..., -0.0754,  0.1962, -0.1019],
        [-0.3610,  0.5741,  0.1023,  ..., -0.0754, -5.0962,  9.8157]],
       dtype=torch.float64)
torch.Size([208480, 60])


In [ ]:
print

In [52]:
#data loader loads data (shocking!)
train_loader = DataLoader(dataset=X_train_tensor, batch_size=40)

In [53]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        #define size of out input (feature number)
        inputSize = X_train_tensor.size(1)
        
        self.flatten = nn.Flatten()
        
        self.linear_relu_stack = nn.Sequential(#The sequential structure of the earthquake classifier
            
            nn.Linear(inputSize, 30), #Bias is on by default if not explicitly set off
            nn.ReLU(),
            nn.Linear(30, 3),
            # nn.ReLU(),
            # nn.Linear(9, 3),
            
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
    # def classifier(self,z):
    #     return nnf.softmax(self(z))

    def parameters(self):
        return self.linear_relu_stack.parameters()

In [54]:
model = NeuralNetwork().double()

In [55]:
criterion = nn.CrossEntropyLoss() #this handles softmax for us so we don't need it in our layers

In [56]:
adamOptimizer = optimizer.Adam(model.parameters(), lr=0.001)

In [57]:
y_train_tensor.size()

torch.Size([83392])

In [58]:
# Train the model
num_epochs = 150
history = {'loss': [], 'val_loss': []}
#training
for epoch in range(num_epochs):
    model.train()
    adamOptimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)  
    loss.backward()
    adamOptimizer.step()
    history['loss'].append(loss.item())
    #testing
    model.eval()
    with torch.no_grad():
        outputs_val = model(X_test_tensor)
        val_loss = criterion(outputs_val, y_test_tensor)  
        history['val_loss'].append(val_loss.item())

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Validation Loss: {val_loss.item():.4f}')

Epoch [10/150], Loss: 1.0116, Validation Loss: 1.0085
Epoch [20/150], Loss: 0.9852, Validation Loss: 0.9824
Epoch [30/150], Loss: 0.9622, Validation Loss: 0.9595
Epoch [40/150], Loss: 0.9411, Validation Loss: 0.9383
Epoch [50/150], Loss: 0.9206, Validation Loss: 0.9178
Epoch [60/150], Loss: 0.9006, Validation Loss: 0.8978
Epoch [70/150], Loss: 0.8818, Validation Loss: 0.8793
Epoch [80/150], Loss: 0.8649, Validation Loss: 0.8628
Epoch [90/150], Loss: 0.8504, Validation Loss: 0.8489
Epoch [100/150], Loss: 0.8384, Validation Loss: 0.8375
Epoch [110/150], Loss: 0.8285, Validation Loss: 0.8282
Epoch [120/150], Loss: 0.8204, Validation Loss: 0.8207
Epoch [130/150], Loss: 0.8138, Validation Loss: 0.8145
Epoch [140/150], Loss: 0.8081, Validation Loss: 0.8093
Epoch [150/150], Loss: 0.8031, Validation Loss: 0.8048


In [89]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split

# ---- Create Dummy DataFrame (60 features + 1 target) ----




# ---- Stratified Train/Test Split ----
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape)
# num_samples = 500

# ---- Custom Dataset ----
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ---- Create datasets ----
train_dataset = TabularDataset(X_train, y_train.to_numpy())
test_dataset = TabularDataset(X_test, y_test.to_numpy())

# ---- DataLoaders ----
#Dataloaders are iterables over the dataset. So when you iterate over it, 
#it will return B randomly from the dataset collected samples (including the data-sample and the target/label), where B is the batch-size.
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ---- Model (60 -> 30 -> 3) ----
model = nn.Sequential(
    # nn.Linear(60, 30),
    # nn.ReLU(),
    # nn.Linear(30, 3)
    
    # nn.Linear(60, 40),
    # nn.ReLU(),
    # nn.Linear(40, 25),
    # nn.ReLU(),
    # nn.Linear(25, 15),
    # nn.ReLU(),
    # nn.Linear(15, 10),
    # nn.ReLU(),
    # nn.Linear(10, 3)

    nn.Linear(60, 40),
    nn.SELU(),
    nn.Linear(40, 25),
    nn.SELU(),
    nn.Linear(25, 15),
    nn.SELU(),
    nn.Linear(15, 10),
    nn.SELU(),
    nn.Linear(10, 3)

    
    
)

# ---- Loss and Optimizer ----
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ---- Training Loop ----
num_epochs = 24

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}")

# ---- Testing Loop ----
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in train_loader:
        outputs = model(batch_x)
        _, predicted = torch.max(outputs, 1)

        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

accuracy = correct / total
print(f"Train Accuracy: {accuracy:.4f}")

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        _, predicted = torch.max(outputs, 1)

        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

(208480, 60)
Epoch 1/24, Loss: 5099.8631
Epoch 2/24, Loss: 4923.3448
Epoch 3/24, Loss: 4953.9684
Epoch 4/24, Loss: 4845.3018
Epoch 5/24, Loss: 4839.6985
Epoch 6/24, Loss: 4991.5707
Epoch 7/24, Loss: 4829.1599
Epoch 8/24, Loss: 4870.4337
Epoch 9/24, Loss: 4818.3440
Epoch 10/24, Loss: 4832.1530
Epoch 11/24, Loss: 5222.5402
Epoch 12/24, Loss: 4803.9130
Epoch 13/24, Loss: 4810.7885
Epoch 14/24, Loss: 4808.9610
Epoch 15/24, Loss: 4786.6769
Epoch 16/24, Loss: 4847.6139
Epoch 17/24, Loss: 5039.5039
Epoch 18/24, Loss: 4825.7581
Epoch 19/24, Loss: 5740.9947
Epoch 20/24, Loss: 4860.9967
Epoch 21/24, Loss: 4997.1000
Epoch 22/24, Loss: 4813.4650
Epoch 23/24, Loss: 4857.6356
Epoch 24/24, Loss: 4865.4871
Train Accuracy: 0.6541
Test Accuracy: 0.6534
